In [ ]:
# Builders for the three game families of Table 2.
import numpy as np

import pygambit as gbt


def create_repeated_game_efg(A, B, T, title=None):
    m, n = A.shape
    if title is None:
        title = f"Repeated {m}x{n} game ({T} periods)"
    g = gbt.Game.new_tree(players=["1", "2"], title=title)
    actions1 = [str(i) for i in range(m)]
    actions2 = [str(i) for i in range(n)]
    payoffs_to_outcomes = {}
    frontier = [(g.root, 0, 0)]
    for t in range(1, T + 1):
        next_frontier = []
        for node, acc_a, acc_b in frontier:
            g.append_move(node, "1", actions1)
            p1_children = list(node.children)
            g.append_move(p1_children, "2", actions2)
            for i, p1_child in enumerate(p1_children):
                for j, child in enumerate(p1_child.children):
                    new_acc_a = acc_a + A[i, j]
                    new_acc_b = acc_b + B[i, j]
                    if t == T:
                        key = (new_acc_a, new_acc_b)
                        if key not in payoffs_to_outcomes:
                            payoffs_to_outcomes[key] = g.add_outcome([new_acc_a, new_acc_b])
                        g.set_outcome(child, payoffs_to_outcomes[key])
                    else:
                        next_frontier.append((child, new_acc_a, new_acc_b))
        frontier = next_frontier
    return g


def create_binary_two_player_efg(level, title=None):
    if title is None:
        title = f"Binary 2-player game (L={level})"
    g = gbt.Game.new_tree(players=["1", "2"], title=title)
    actions = ["L", "R"]
    g.append_move(g.root, "1", actions)
    frontier = [g.root]
    for depth in range(1, level):
        player_label = str((depth % 2) + 1)
        new_frontier = []
        for parent in frontier:
            children = list(parent.children)
            g.append_move(children, player_label, actions)
            new_frontier.extend(children)
        frontier = new_frontier
    zero_outcome = g.add_outcome([0, 0])
    for parent in frontier:
        for leaf in parent.children:
            g.set_outcome(leaf, zero_outcome)
    return g


def create_centipede_efg(N, title=None):
    if title is None:
        title = f"Centipede game ({N} rounds)"
    g = gbt.Game.new_tree(players=["1", "2"], title=title)
    actions = ["Take", "Push"]
    zero_outcome = g.add_outcome([0, 0])
    current = g.root
    for t in range(N):
        player = str((t % 2) + 1)
        g.append_move(current, player, actions)
        take_child, push_child = current.children
        g.set_outcome(take_child, zero_outcome)
        if t == N - 1:
            g.set_outcome(push_child, zero_outcome)
        else:
            current = push_child
    return g

In [ ]:
# Gambit 16.5: the previous subgame-root procedure on one game from each family,
# timed the way Table 2 times Algorithm 1: a fresh game per run, built off the
# clock, median over the runs. Requires pygambit 16.5 (pip install pygambit==16.5.0);
# run the builders cell first.
#
# Gambit 16.5 has no call that returns every subgame root, so finding them all
# means enumerating the nodes and testing each one with Node.is_subgame_root,
# which rejects terminal nodes and nodes in multi-member infosets at once and
# certifies any other decision node by scanning every infoset of the game, with
# an ancestor walk per member. Centipede 3k takes about 19 s per run on an
# Apple M1 Pro, so its 1,000 runs take about 5.5 hours; progress is printed every 50 runs.
import statistics
import time

assert gbt.__version__.startswith("16.5."), "Needs pygambit 16.5: pip install pygambit==16.5.0"
print(f"pygambit {gbt.__version__}\n")

_Z23 = np.zeros((2, 3), dtype=int)
GAMES = [  # (name, builder, runs, Table 2 median of Algorithm 1 in ms on Apple M1 Pro)
    ("binary-2p-depth-10", lambda: create_binary_two_player_efg(level=10),    1000, 1.251),
    ("repeated-2x3-T4",    lambda: create_repeated_game_efg(_Z23, _Z23, T=4), 1000, 1.186),
    ("centipede-N3000",    lambda: create_centipede_efg(N=3000),              1000, 3.735),
]


def fmt(seconds, ref=None):
    ref = seconds if ref is None else ref
    return f"{seconds:.2f} s" if ref >= 1 else f"{seconds * 1e3:.3f} ms"


for name, build, n_runs, table2_ms in GAMES:
    visit, test, total = [], [], []
    started = time.perf_counter()
    for i in range(1, n_runs + 1):
        game = build()
        t0 = time.perf_counter()
        nodes = list(game.nodes)                                   # enumerate every node
        t1 = time.perf_counter()
        roots = [node for node in nodes if node.is_subgame_root]   # test every node
        t2 = time.perf_counter()
        visit.append(t1 - t0)
        test.append(t2 - t1)
        total.append(t2 - t0)
        n_nodes, n_roots = len(nodes), len(roots)
        del game, nodes, roots                                     # off the clock, as in Table 2
        if i % 50 == 0 and i < n_runs and time.perf_counter() - started > 60:
            left = (time.perf_counter() - started) / i * (n_runs - i)
            print(f"  {name}: {i}/{n_runs} runs, median so far {fmt(statistics.median(total))},"
                  f" about {left / 3600:.1f} h left", flush=True)
    total.sort()
    t = statistics.median(total)
    # semi-interquartile range, as in Table 2
    siqr = (total[3 * n_runs // 4] - total[n_runs // 4]) / 2
    ratio = t * 1e3 / table2_ms
    times = f"{ratio:,.0f}x" if ratio >= 10 else f"{ratio:.1f}x"
    print(f"{name}  (nodes={n_nodes:,}, subgame roots={n_roots:,})")
    print(f"  Gambit 16.5, median of {n_runs:,} runs: {fmt(t)} ± {fmt(siqr, t)}  "
          f"(enumerating {fmt(statistics.median(visit), t)}, "
          f"testing {fmt(statistics.median(test), t)})")
    print(f"  Table 2 (Apple M1 Pro), Algorithm 1: {table2_ms:.3f} ms, "
          f"so 16.5 is {times} slower\n")